# 1MTR58 – LAB2 – Experiencia 1: Eye Tracking + EAR

En esta parte del laboratorio se trabajará con la detección de parpadeos a partir del **EAR (Eye Aspect Ratio)**.

Ejecutar las celdas en orden. En cada prueba primero se realiza una calibración y luego se registra la señal durante 20 s.

Durante el registro se muestra en la misma ventana:
- cámara y landmarks de los ojos;
- valor de EAR;
- umbral calibrado;
- número de parpadeos y PERCLOS;
- gráfica del EAR en tiempo real.

Al finalizar cada prueba, el notebook guarda automáticamente un archivo `.png` con la ventana mostrada y un archivo `.csv` con los datos en `resultados_eye_tracking/`.


## Verificación del entorno

Ejecute esta celda antes de comenzar.

El notebook requiere **Python 3.11 o 3.12**. Si falta alguna librería, se instalará automáticamente en el mismo entorno de Python que está usando el notebook.

Si se instala alguna dependencia, reinicie el kernel y vuelva a ejecutar el notebook desde el inicio.


In [ ]:
import sys
import subprocess
import importlib.util

print("Python:", sys.version)
print("Entorno:", sys.executable)

if not ((3, 11) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(
        "Este laboratorio requiere Python 3.11 o 3.12. "
        "Seleccione un kernel compatible antes de continuar."
    )

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "cv2": "opencv-python",
    "mediapipe": "mediapipe==0.10.21",
}

to_install = []

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        to_install.append(package)

try:
    import mediapipe as mp
    if mp.__version__ != "0.10.21":
        to_install.append("mediapipe==0.10.21")
except Exception:
    if "mediapipe==0.10.21" not in to_install:
        to_install.append("mediapipe==0.10.21")

to_install = list(dict.fromkeys(to_install))

if to_install:
    print("Instalando dependencias:", to_install)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *to_install]
    )
    print("\nInstalación completada.")
    print("Reinicie el kernel y vuelva a ejecutar el notebook desde el inicio.")
else:
    print("✓ Entorno listo. Puede continuar con el laboratorio.")


## 0. Preparación

Ejecutar esta sección una vez antes de comenzar las pruebas.


In [ ]:

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from pathlib import Path
from datetime import datetime
from IPython.display import display
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from collections import deque

# Evita que Matplotlib abra una segunda ventana:
# la gráfica se renderiza y se inserta debajo del video de OpenCV.
plt.ioff()

OUTPUT_DIR = Path("resultados_eye_tracking")
OUTPUT_DIR.mkdir(exist_ok=True)

CAMERA_INDEX = 0

LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

CALIBRATION_TIME = 4       # s
THRESHOLD_K = 1.25
THRESHOLD_SCALE = 0.68
THRESHOLD_MIN = 0.08
THRESHOLD_MAX = 0.45

# Visualización en tiempo real
LIVE_PLOT_HEIGHT = 220
LIVE_PLOT_POINTS = 120     # cantidad máxima de muestras visibles
PERCLOS_WINDOW_SEC = 60

mp_face_mesh = mp.solutions.face_mesh
print("Preparación completada.")


### 0.1 Prueba de cámara

Ejecutar esta celda para comprobar que la cámara esté funcionando correctamente.


In [ ]:
def camera_test(camera_index=CAMERA_INDEX):
    """Prueba la cámara. Presionar q para cerrar."""
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(
            f"No se pudo abrir la cámara {camera_index}. "
            "Prueba CAMERA_INDEX = 1 u otro índice."
        )

    while True:
        ret, frame = cap.read()
        if not ret:
            print("No se pudo leer un frame.")
            break

        cv2.putText(frame, "Prueba de camara - q para salir",
                    (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0, 255, 255), 2)
        cv2.imshow("Prueba de camara", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# Descomentar para probar:
camera_test()

## Funciones

Funciones utilizadas para calcular el EAR, realizar la calibración y mostrar la señal en tiempo real.


In [ ]:
def get_eye_aspect_ratio(landmarks, eye_points):
    """
    Calcula el Eye Aspect Ratio (EAR) usando 6 landmarks.
    EAR = (distancia vertical 1 + distancia vertical 2) / (2 * distancia horizontal)
    """
    points = [landmarks[i] for i in eye_points]

    vertical_1 = np.linalg.norm(
        np.array([points[1].x, points[1].y]) -
        np.array([points[5].x, points[5].y])
    )
    vertical_2 = np.linalg.norm(
        np.array([points[2].x, points[2].y]) -
        np.array([points[4].x, points[4].y])
    )
    horizontal = np.linalg.norm(
        np.array([points[0].x, points[0].y]) -
        np.array([points[3].x, points[3].y])
    )

    if horizontal == 0:
        return np.nan

    return (vertical_1 + vertical_2) / (2.0 * horizontal)


def calculate_threshold(calibration_ears):
    """
    Regla conservada del código base:
    EAR_THRESHOLD = (media - 1.25*std) * 0.68, limitado a [0.08, 0.45].
    """
    values = np.asarray(calibration_ears, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("No se obtuvieron valores EAR válidos durante la calibración.")

    mu = float(np.mean(values))
    sd = float(np.std(values))
    threshold = (mu - THRESHOLD_K * sd) * THRESHOLD_SCALE
    threshold = float(np.clip(threshold, THRESHOLD_MIN, THRESHOLD_MAX))
    return threshold, mu, sd


def _safe_name(text):
    text = str(text).strip().replace(" ", "_")
    return "".join(ch for ch in text if ch.isalnum() or ch in "_-")


class LiveEARPlot:
    """
    Gráfica EAR embebida dentro de la ventana de OpenCV.
    Inspirada en eyetracking_biomecatronica:
    se actualiza simultáneamente con la cámara.
    """
    def __init__(self, max_points=LIVE_PLOT_POINTS):
        self.max_points = max_points
        self.x_values = deque(maxlen=max_points)
        self.ear_values = deque(maxlen=max_points)

        self.fig, self.ax = plt.subplots(figsize=(7, 2.5), dpi=100)
        self.fig.patch.set_facecolor("black")
        self.ax.set_facecolor("black")

        self.ear_curve, = self.ax.plot(
            [], [], color="#56f10d", linewidth=2, label="EAR"
        )
        self.threshold_line, = self.ax.plot(
            [], [], color="#f70202", linestyle="--", linewidth=2, label="Umbral"
        )

        self.ax.set_ylim(0, 0.45)
        self.ax.set_xlabel("Frames", color="white")
        self.ax.set_ylabel("EAR", color="white")
        self.ax.tick_params(colors="white")
        for spine in self.ax.spines.values():
            spine.set_color("white")

        self.ax.legend(loc="upper right")
        self.fig.tight_layout()
        self.canvas = FigureCanvas(self.fig)

    def update(self, ear, frame_number, threshold=None):
        if np.isfinite(ear):
            self.x_values.append(frame_number)
            self.ear_values.append(float(ear))

        x = list(self.x_values)
        y = list(self.ear_values)

        if x:
            self.ear_curve.set_xdata(x)
            self.ear_curve.set_ydata(y)

            threshold_value = float(threshold) if threshold is not None else 0.0
            self.threshold_line.set_xdata(x)
            self.threshold_line.set_ydata([threshold_value] * len(x))

            if len(x) == 1:
                self.ax.set_xlim(max(0, x[0] - 1), x[0] + 1)
            else:
                self.ax.set_xlim(x[0], max(x[-1], x[0] + 1))

            # Ajuste vertical conservador para evitar cortar el EAR.
            visible_max = max(y) if y else 0.4
            self.ax.set_ylim(0, max(0.45, visible_max * 1.10))

        self.canvas.draw()
        buf = np.asarray(self.canvas.buffer_rgba())
        return cv2.cvtColor(buf, cv2.COLOR_RGBA2BGR)

    def close(self):
        plt.close(self.fig)


def _draw_eye_landmarks(frame, face_landmarks):
    for idx in LEFT_EYE + RIGHT_EYE:
        lm = face_landmarks.landmark[idx]
        x = int(lm.x * frame.shape[1])
        y = int(lm.y * frame.shape[0])
        cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)


def _combine_camera_and_plot(frame, plot_img):
    """Iguala anchos y coloca la gráfica debajo del video."""
    plot_img = cv2.resize(
        plot_img,
        (frame.shape[1], LIVE_PLOT_HEIGHT),
        interpolation=cv2.INTER_AREA
    )
    return cv2.vconcat([frame, plot_img])


def run_ear_session(
    condition_name,
    calibration_time=CALIBRATION_TIME,
    record_time=20,
    camera_index=CAMERA_INDEX,
    save_outputs=True,
    participant_name=None,
):
    """
    Flujo de una sesión:
    1) calibración del EAR;
    2) cálculo del EAR_THRESHOLD;
    3) registro;
    4) detección de parpadeos y PERCLOS;
    5) cámara + gráfica EAR simultáneas en la MISMA ventana.

    Al finalizar el registro se guarda automáticamente un PNG con la ventana mostrada, además del CSV.
    Presionar q para terminar.
    """
    condition_name = str(condition_name)

    if participant_name is None:
        participant_name = input("Nombre o código del integrante: " ).strip()

    participant_name = participant_name.strip() if participant_name else "sin_nombre"
    file_tag = _safe_name(f"{condition_name}_{participant_name}")
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(
            f"No se pudo abrir la cámara {camera_index}. "
            "Prueba cambiando CAMERA_INDEX."
        )

    face_mesh = mp_face_mesh.FaceMesh(
        refine_landmarks=True,
        max_num_faces=1
    )

    live_plot = LiveEARPlot()

    calib_ears = []
    records = []

    frame_number = 0
    total_frames = 0
    blink_count = 0
    blink_detected = False
    user_quit = False
    last_capture = None

    # Para PERCLOS móvil, igual que el notebook de referencia.
    perclos_window = deque()
    time_window = deque()

    window_name = f"LAB2 - {condition_name} | EAR + Calibracion + PERCLOS"

    try:
        # ==============================================================
        # ETAPA 1: CALIBRACIÓN
        # ==============================================================
        print(f"\n[{condition_name}] Integrante: {participant_name}")
        print(f"[{condition_name}] CALIBRACIÓN: mantén los ojos abiertos.")
        calibration_start = time.time()

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb)

            ear = np.nan

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                left_ear = get_eye_aspect_ratio(
                    face_landmarks.landmark, LEFT_EYE
                )
                right_ear = get_eye_aspect_ratio(
                    face_landmarks.landmark, RIGHT_EYE
                )
                ear = float(np.nanmean([left_ear, right_ear]))

                _draw_eye_landmarks(frame, face_landmarks)

                if np.isfinite(ear):
                    calib_ears.append(ear)

            elapsed = time.time() - calibration_start
            remaining = max(0, calibration_time - elapsed)

            cv2.putText(
                frame,
                f"CALIBRANDO: ojos abiertos {remaining:0.1f}s",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.72,
                (0, 255, 255),
                2,
            )

            cv2.putText(
                frame,
                f"EAR: {ear:.3f}" if np.isfinite(ear) else "EAR: --",
                (20, 78),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.72,
                (255, 255, 0),
                2,
            )

            # Durante calibración ya se construye la curva EAR.
            plot_img = live_plot.update(
                ear=ear,
                frame_number=frame_number,
                threshold=None
            )

            combined = _combine_camera_and_plot(frame, plot_img)
            cv2.imshow(window_name, combined)

            frame_number += 1

            if cv2.waitKey(1) & 0xFF == ord("q"):
                user_quit = True
                break

            if elapsed >= calibration_time:
                break

        if user_quit:
            print(f"[{condition_name}] Sesión cancelada durante la calibración.")
            return None, pd.DataFrame()

        if len(calib_ears) < 5:
            raise RuntimeError(
                "La calibración obtuvo muy pocos valores EAR. "
                "Repite con el rostro visible y buena iluminación."
            )

        ear_threshold, calib_mean, calib_std = calculate_threshold(calib_ears)

        print(f"[{condition_name}] EAR medio calibración = {calib_mean:.3f}")
        print(f"[{condition_name}] Desv. estándar = {calib_std:.3f}")
        print(f"[{condition_name}] EAR_THRESHOLD = {ear_threshold:.3f}")

        # ==============================================================
        # ETAPA 2: REGISTRO
        # ==============================================================
        print(
            f"[{condition_name}] REGISTRO: {record_time} s. "
            "Parpadea normalmente. La imagen PNG se guardará automáticamente."
        )

        recording_start = time.time()

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            elapsed = time.time() - recording_start
            total_frames += 1

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb)

            ear = np.nan
            below_threshold = False

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                left_ear = get_eye_aspect_ratio(
                    face_landmarks.landmark, LEFT_EYE
                )
                right_ear = get_eye_aspect_ratio(
                    face_landmarks.landmark, RIGHT_EYE
                )
                ear = float(np.nanmean([left_ear, right_ear]))

                _draw_eye_landmarks(frame, face_landmarks)

                if np.isfinite(ear):
                    below_threshold = ear < ear_threshold

                    # Un parpadeo se cuenta una sola vez al entrar
                    # en la región EAR < threshold.
                    if below_threshold:
                        if not blink_detected:
                            blink_count += 1
                            blink_detected = True
                    else:
                        blink_detected = False

                    now = time.time()
                    time_window.append(now)
                    perclos_window.append(1 if below_threshold else 0)

                    while (
                        time_window and
                        (now - time_window[0]) > PERCLOS_WINDOW_SEC
                    ):
                        time_window.popleft()
                        perclos_window.popleft()

                    records.append({
                        "time_s": elapsed,
                        "EAR": ear,
                        "below_threshold": int(below_threshold),
                    })

            perclos = (
                100 * sum(perclos_window) / len(perclos_window)
                if perclos_window else np.nan
            )

            # ----------------------------------------------------------
            # Información visible en la cámara
            # ----------------------------------------------------------
            cv2.putText(
                frame,
                f"Parpadeos: {blink_count}",
                (20, 42),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.90,
                (255, 0, 0),
                2,
            )

            cv2.putText(
                frame,
                f"PERCLOS({PERCLOS_WINDOW_SEC}s): {perclos:0.1f}%"
                if np.isfinite(perclos)
                else f"PERCLOS({PERCLOS_WINDOW_SEC}s): --",
                (20, 82),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (0, 200, 255),
                2,
            )

            cv2.putText(
                frame,
                f"EAR: {ear:.3f}" if np.isfinite(ear) else "EAR: --",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.70,
                (255, 255, 0),
                2,
            )

            cv2.putText(
                frame,
                f"EAR_THRESHOLD: {ear_threshold:.3f}",
                (20, 158),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.70,
                (0, 0, 255),
                2,
            )

            cv2.putText(
                frame,
                f"Tiempo: {elapsed:0.1f}/{record_time}s",
                (20, 196),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.62,
                (0, 0, 0),
                2,
            )

            cv2.putText(
                frame,
                "PNG + CSV se guardan automaticamente | q: salir",
                (20, frame.shape[0] - 18),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                (0, 0, 0),
                1,
            )

            # Gráfica en vivo con el umbral ya calibrado.
            plot_img = live_plot.update(
                ear=ear,
                frame_number=frame_number,
                threshold=ear_threshold
            )

            combined = _combine_camera_and_plot(frame, plot_img)
            last_capture = combined.copy()
            cv2.imshow(window_name, combined)

            frame_number += 1

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

            if elapsed >= record_time:
                break

    finally:
        cap.release()
        face_mesh.close()
        live_plot.close()
        cv2.destroyAllWindows()

    if len(records) == 0:
        raise RuntimeError("No se registraron valores EAR válidos.")

    df = pd.DataFrame(records)

    # Para registros de 20 s, este PERCLOS coincide con la ventana móvil
    # de 60 s porque toda la sesión cabe dentro de la ventana.
    perclos_session = float(df["below_threshold"].mean() * 100)

    result = {
        "integrante": participant_name,
        "condicion": condition_name,
        "EAR_THRESHOLD": ear_threshold,
        "EAR_calibracion_media": calib_mean,
        "EAR_calibracion_std": calib_std,
        "PERCLOS_pct": perclos_session,
        "parpadeos": int(blink_count),
        "duracion_registro_s": float(df["time_s"].max()),
        "frames_validos": int(len(df)),
        "frames_totales": int(total_frames),
        "deteccion_landmarks_pct": float(
            100 * len(df) / max(1, total_frames)
        ),
    }

    if save_outputs:
        csv_path = OUTPUT_DIR / f"{stamp}_{file_tag}.csv"
        df.to_csv(csv_path, index=False)
        print(f"CSV guardado: {csv_path}")

        if last_capture is not None:
            png_path = OUTPUT_DIR / f"{stamp}_{file_tag}.png"
            ok = cv2.imwrite(str(png_path), last_capture)
            if ok:
                result["archivo_png"] = str(png_path)
                print(f"PNG guardado: {png_path}")
            else:
                print("Advertencia: no se pudo guardar la imagen PNG.")

    display(pd.DataFrame([result]))
    return result, df


# 1. Experiencia 1 – Detección de parpadeo y evaluación de robustez

## 1.1 Detección y calibración del EAR

**Cada integrante** debe realizar esta prueba una vez, en posición frontal y sin lentes.

Ejecute la celda una vez por integrante. Al finalizar cada ejecución se guardará automáticamente un `.png` con la cámara, la señal EAR y el umbral obtenido.


In [ ]:
# 1.1 – Ejecutar ESTA CELDA una vez por integrante
integrante_11 = input("Nombre o código del integrante: ").strip()

resultado_11, datos_11 = run_ear_session(
    condition_name="1.1_Frontal_sin_lentes",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_11,
)


### Para el reporte

Adjuntar el `.png` generado para cada integrante y describir brevemente qué ocurre con el valor de EAR durante un parpadeo.


## 1.2 Aplicación a condiciones reales

### 1.2.1 Condición base: calibración y umbral de parpadeo

Esta prueba la realiza el integrante asignado a las secciones **1.2.1 y 1.2.2**. Registrar el `EAR_THRESHOLD` obtenido y usar el `.png` generado automáticamente.


In [ ]:
# 1.2.1 – Condición base
integrante_a = input("Nombre o código del integrante asignado a 1.2.1 y 1.2.2: ").strip()

resultado_base, datos_base = run_ear_session(
    condition_name="1.2.1_Base_frontal_sin_lentes",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_a,
)


### 1.2.2 Efecto de los lentes

Repetir la prueba utilizando lentes.

Comparar el nuevo `EAR_THRESHOLD` con el obtenido en la condición base y observar si cambia el comportamiento de la detección.


In [ ]:
# 1.2.2 – Ejecutar esta celda usando lentes
resultado_lentes, datos_lentes = run_ear_session(
    condition_name="1.2.2_Con_lentes",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_a,
)

# Comparación directa de umbrales
comparacion_lentes = pd.DataFrame([
    {
        "condicion": "Base sin lentes",
        "EAR_THRESHOLD": resultado_base["EAR_THRESHOLD"],
        "PERCLOS_pct": resultado_base["PERCLOS_pct"],
        "parpadeos": resultado_base["parpadeos"],
    },
    {
        "condicion": "Con lentes",
        "EAR_THRESHOLD": resultado_lentes["EAR_THRESHOLD"],
        "PERCLOS_pct": resultado_lentes["PERCLOS_pct"],
        "parpadeos": resultado_lentes["parpadeos"],
    },
])
display(comparacion_lentes)

delta_umbral = (
    resultado_lentes["EAR_THRESHOLD"] -
    resultado_base["EAR_THRESHOLD"]
)
print(f"Cambio de EAR_THRESHOLD (lentes - base): {delta_umbral:+.3f}")

### 1.2.3 Efecto del ángulo de cámara

Esta sección la realiza otro integrante. Realizar la prueba en tres posiciones de cámara:

1. superior;
2. frontal;
3. inferior.

Mover la cámara antes de ejecutar cada celda. Se guardará automáticamente un `.png` por cada posición.


In [ ]:
# 1.2.3A – Cámara en posición SUPERIOR
integrante_b = input("Nombre o código del integrante asignado a 1.2.3: ").strip()

resultado_superior, datos_superior = run_ear_session(
    condition_name="1.2.3_Superior",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_b,
)


In [ ]:
# 1.2.3B – Cámara en posición FRONTAL
resultado_frontal, datos_frontal = run_ear_session(
    condition_name="1.2.3_Frontal",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_b,
)

In [ ]:
# 1.2.3C – Cámara en posición INFERIOR
resultado_inferior, datos_inferior = run_ear_session(
    condition_name="1.2.3_Inferior",
    calibration_time=4,
    record_time=20,
    participant_name=integrante_b,
)

In [ ]:
# Comparación 1.2.3
comparacion_angulos = pd.DataFrame([
    {
        "posicion": "Superior",
        "EAR_THRESHOLD": resultado_superior["EAR_THRESHOLD"],
        "PERCLOS_pct": resultado_superior["PERCLOS_pct"],
        "parpadeos": resultado_superior["parpadeos"],
        "frames_validos": resultado_superior["frames_validos"],
        "deteccion_landmarks_pct": resultado_superior["deteccion_landmarks_pct"],
    },
    {
        "posicion": "Frontal",
        "EAR_THRESHOLD": resultado_frontal["EAR_THRESHOLD"],
        "PERCLOS_pct": resultado_frontal["PERCLOS_pct"],
        "parpadeos": resultado_frontal["parpadeos"],
        "frames_validos": resultado_frontal["frames_validos"],
        "deteccion_landmarks_pct": resultado_frontal["deteccion_landmarks_pct"],
    },
    {
        "posicion": "Inferior",
        "EAR_THRESHOLD": resultado_inferior["EAR_THRESHOLD"],
        "PERCLOS_pct": resultado_inferior["PERCLOS_pct"],
        "parpadeos": resultado_inferior["parpadeos"],
        "frames_validos": resultado_inferior["frames_validos"],
        "deteccion_landmarks_pct": resultado_inferior["deteccion_landmarks_pct"],
    },
])

display(comparacion_angulos)
print(
    "Rango de umbrales por ángulo:",
    f"{comparacion_angulos['EAR_THRESHOLD'].min():.3f}",
    "a",
    f"{comparacion_angulos['EAR_THRESHOLD'].max():.3f}",
)

Comparar los tres valores de `EAR_THRESHOLD` y revisar en qué posición los landmarks presentan mayor pérdida o inestabilidad.


### 1.2.4 Comparación entre integrantes

Cada integrante debe realizar un registro de **20 s**.

Primero inicializar la lista de resultados y luego ejecutar la siguiente celda una vez por integrante. Al final se muestran los valores de EAR threshold, PERCLOS y número de parpadeos para compararlos. Cada ejecución guarda automáticamente su `.png` y `.csv`.


In [ ]:
# Ejecutar una vez antes de comenzar el registro del grupo
resultados_grupo = []

In [ ]:
# Ejecutar ESTA CELDA una vez por integrante
integrante = input("Nombre o código del integrante: ").strip()

resultado_integrante, datos_integrante = run_ear_session(
    condition_name="1.2.4_Comparacion_integrantes",
    calibration_time=4,
    record_time=20,
    participant_name=integrante,
)

resultados_grupo.append(resultado_integrante)

tabla_grupo = pd.DataFrame(resultados_grupo)
display(
    tabla_grupo[
        ["condicion", "EAR_THRESHOLD", "PERCLOS_pct", "parpadeos", "frames_validos", "deteccion_landmarks_pct"]
    ]
)

In [ ]:
# Resumen final del grupo para 1.2.4
tabla_grupo = pd.DataFrame(resultados_grupo)

if len(tabla_grupo) == 0:
    print("Aún no se han registrado integrantes.")
else:
    display(
        tabla_grupo[
            ["condicion", "EAR_THRESHOLD", "PERCLOS_pct", "parpadeos", "frames_validos", "deteccion_landmarks_pct"]
        ]
    )

    print(
        f"Rango EAR_THRESHOLD del grupo: "
        f"{tabla_grupo['EAR_THRESHOLD'].min():.3f} – "
        f"{tabla_grupo['EAR_THRESHOLD'].max():.3f}"
    )
    print(
        f"Rango PERCLOS: "
        f"{tabla_grupo['PERCLOS_pct'].min():.1f}% – "
        f"{tabla_grupo['PERCLOS_pct'].max():.1f}%"
    )
    print(
        f"Rango de parpadeos en 20 s: "
        f"{int(tabla_grupo['parpadeos'].min())} – "
        f"{int(tabla_grupo['parpadeos'].max())}"
    )

## Archivos generados

Cada ejecución guarda automáticamente en:

`resultados_eye_tracking/`

- un `.png` con la evidencia visual de la prueba;
- un `.csv` con los valores EAR registrados.

Para el reporte, usar los archivos `.png` correspondientes a cada prueba.
